# LWSCE pilot 3 on Colab

CIFAR100, ResNet-18 (random crop + flip, AdamW, cosine schedule, mixed precision), lr 3e-4 (picked by both CE and LWSCE in pilot 2).
LWSCE now keeps the most probable non-target class unmasked, which fixes the collapse of alpha >= 0.5.

* **Stage 1** (30 epochs): LWSCE with alpha 0.2 / 0.5 / 1.0 / 2.0, label smoothing and LWSCE + label smoothing, against CE.
  The CE runs of pilot 2 are reused when they are on Drive (CE did not change).
* **Stage 2** (short budgets): CE vs LWSCE at its selected alpha, trained for 10 and 15 epochs with the cosine schedule fitted to that budget.
* **Stage 3** (optional, off by default): CE vs LWSCE with ResNet-50.

Each loss picks its configuration on the validation set, then test accuracy is compared.

1. **Runtime → Change runtime type → GPU** (T4 is enough).
2. **Runtime → Run all.** Google Drive will ask for permission once.
3. Results and models are saved to Google Drive (`MyDrive/label_mask_loss_runs`). If the session disconnects, just **Run all** again: finished configurations are skipped (a configuration that was interrupted midway restarts from its first run).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU"
print("torch", torch.__version__)

## 1. Experiment settings

In [ ]:
# where the code comes from: "github" clones BRANCH, "zip" asks you to upload a zip of the repo
CODE_SOURCE = "github"
REPO_URL = "https://github.com/machine-learning-is-easy/label_mask_loss.git"
BRANCH = "revert"

NAME = "pilot3"     # prefix of the result names, change it to start a fresh sweep
REUSE_NAME = "pilot2"  # CE runs of this sweep are copied instead of trained again
DATASET = "CIFAR100"
MODEL = "resnet18"
OPT_ALG = "ADAMW"
LRS = ["3e-4"]
LOSSES = ["CROSSENTROPY", "LABELSMOOTHING", "LWSCE", "LWSCE+LS"]  # the first one is the baseline of the comparisons
LWSCE_ALPHAS = ["0.2", "0.5", "1.0", "2.0"]
LWSCE_LS_ALPHAS = ["0.2"]
SHORT_EPOCHS = [11, 16]  # stage 2: 10 and 15 epochs, set to [] to skip
RUN_RESNET50 = False     # stage 3, about 2.5x the time of a ResNet-18 run
RUNS = 3
EPOCH = 31          # training.py trains EPOCH - 1 epochs
WEIGHT_DECAY = "5e-2"
SCHEDULER = "cosine"

DRIVE_DIR = "/content/drive/MyDrive/label_mask_loss_runs"
CODE_DIR = "/content/label_mask_loss"

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(os.path.join(DRIVE_DIR, "result"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, "model"), exist_ok=True)

## 3. Get the code

In [ ]:
import os, shutil, subprocess, zipfile

if CODE_SOURCE == "github":
    if os.path.exists(CODE_DIR):
        subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
                        REPO_URL, CODE_DIR], check=True)
elif CODE_SOURCE == "zip":
    from google.colab import files
    uploaded = files.upload()  # a zip of the repo containing training.py, pilot.py, loss/, model_define/, cv/
    zip_name = next(iter(uploaded))
    shutil.rmtree(CODE_DIR, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(CODE_DIR)
    # accept a zip whose content sits inside one top-level folder
    entries = os.listdir(CODE_DIR)
    if "training.py" not in entries and len(entries) == 1:
        inner = os.path.join(CODE_DIR, entries[0])
        for name in os.listdir(inner):
            shutil.move(os.path.join(inner, name), CODE_DIR)
        os.rmdir(inner)
else:
    raise ValueError(CODE_SOURCE)

for f in ["training.py", "pilot.py", "loss/partial_loss.py", "model_define/resnet.py"]:
    assert os.path.exists(os.path.join(CODE_DIR, f)), "missing " + f + ", is the branch / zip up to date?"
assert "keep_hardest" in open(os.path.join(CODE_DIR, "loss/partial_loss.py")).read(), "old code, pull the latest " + BRANCH

# keep results and models on Drive so they survive a disconnect
for name in ["result", "model"]:
    path = os.path.join(CODE_DIR, name)
    if os.path.islink(path):
        continue
    shutil.rmtree(path, ignore_errors=True)
    os.symlink(os.path.join(DRIVE_DIR, name), path)

%cd {CODE_DIR}
!git log --oneline -1 2>/dev/null || true

## 4. Run the sweeps

Every run prints one line per epoch; multiply the time of the first epochs by the number of epochs, runs and configurations to estimate the total.

In [ ]:
import os
import shutil
import pandas as pd

def run_dir(name, model, loss, lr, alpha=None):
    # same naming as pilot.py / training.py
    tag = "{}_lr{}".format(name, lr) + ("_a{}".format(alpha) if alpha else "")
    return os.path.join("result", "{}_{}_{}_{}_{}_result".format(model, DATASET, OPT_ALG, loss, tag))

def sweep_path(name, model):
    return os.path.join("result", "{}_{}_{}_{}_sweep.csv".format(name, DATASET, model, OPT_ALG))

def pilot(name, model, losses, lrs, alphas, epoch=EPOCH, summarize_only=False):
    cmd = ("PYTHONUNBUFFERED=1 python pilot.py --name {} --dataset {} --model {} --opt_alg {} --losses {} --lrs {} "
           "--alphas {} --runs {} --epoch {} --weight_decay {} --scheduler {}").format(
        name, DATASET, model, OPT_ALG, " ".join(losses), " ".join(lrs), " ".join(alphas), RUNS, epoch,
        WEIGHT_DECAY, SCHEDULER)
    if summarize_only:
        cmd += " --summarize_only"
    get_ipython().system(cmd)
    return pd.read_csv(sweep_path(name, model), dtype=str)

def selected(sweep, loss):
    row = sweep[(sweep["loss"] == loss) & (sweep["selected"] == "True")].iloc[0]
    return row["lr"], row["alpha"] if isinstance(row["alpha"], str) else None

ALL_ALPHAS = sorted(set(LWSCE_ALPHAS + LWSCE_LS_ALPHAS), key=float)

### Stage 1: alpha, label smoothing and LWSCE + label smoothing (30 epochs)

In [ ]:
# CE did not change since REUSE_NAME, copy its runs instead of training them again
for lr in LRS:
    src, dst = run_dir(REUSE_NAME, MODEL, "CROSSENTROPY", lr), run_dir(NAME, MODEL, "CROSSENTROPY", lr)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copytree(src, dst)
        print("reused", src)

baselines = [loss for loss in LOSSES if loss in ("CROSSENTROPY", "LABELSMOOTHING")]
pilot(NAME, MODEL, baselines, LRS, ["0.2"])  # alpha is not used by these losses
if "LWSCE" in LOSSES:
    pilot(NAME, MODEL, ["LWSCE"], LRS, LWSCE_ALPHAS)
if "LWSCE+LS" in LOSSES:
    pilot(NAME, MODEL, ["LWSCE+LS"], LRS, LWSCE_LS_ALPHAS)
# one summary, so each loss picks its configuration among everything trained
sweep = pilot(NAME, MODEL, LOSSES, LRS, ALL_ALPHAS, summarize_only=True)

### Stage 2: short training budgets, CE vs LWSCE at its selected alpha

In [ ]:
ce_lr, _ = selected(sweep, "CROSSENTROPY")
lw_lr, lw_alpha = selected(sweep, "LWSCE")
for epoch in SHORT_EPOCHS:
    name = "{}_e{}".format(NAME, epoch - 1)
    print("budget {} epochs: CE lr {}, LWSCE lr {} alpha {}".format(epoch - 1, ce_lr, lw_lr, lw_alpha))
    pilot(name, MODEL, ["CROSSENTROPY"], [ce_lr], ["0.2"], epoch=epoch)
    pilot(name, MODEL, ["LWSCE"], [lw_lr], [lw_alpha], epoch=epoch)
    pilot(name, MODEL, ["CROSSENTROPY", "LWSCE"], sorted({ce_lr, lw_lr}), [lw_alpha], epoch=epoch, summarize_only=True)

### Stage 3 (optional): ResNet-50, CE vs LWSCE at the configurations selected with ResNet-18

In [ ]:
if RUN_RESNET50:
    picked = {loss: selected(sweep, loss) for loss in ["CROSSENTROPY", "LWSCE"]}
    for loss, (lr, alpha) in picked.items():
        pilot(NAME, "resnet50", [loss], [lr], [alpha or "0.2"])
    lrs = sorted({lr for lr, _ in picked.values()})
    pilot(NAME, "resnet50", list(picked), lrs, [picked["LWSCE"][1]], summarize_only=True)
else:
    print("skipped, set RUN_RESNET50 = True to run it")

## 5. Results

All numbers are accuracies in %. Each run's test accuracy is taken at its best validation epoch; tables and figures are also saved to `MyDrive/label_mask_loss_runs/result`.

* **Sweep:** mean over runs; the highlighted row of each loss is its configuration chosen by **validation** accuracy.
* **Comparison:** runs share the data split and the seed of each run, so run *i* of every loss is paired. `wins` counts runs where the loss beats CE; `p` is a paired t-test. With only a few runs, p-values are a rough signal.
* **Curves:** mean over runs, the band is ± one standard deviation. `low_prob_ratio` is the share of non-target classes with probability ≤ 2·alpha/C (LWSCE masks them, except the most probable one; for the other losses it shows what LWSCE would mask). A `logit_std` near 0 means the network outputs a constant, i.e. training collapsed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from scipy import stats
from IPython.display import display

CURVES = [("val_acc", "Validation accuracy (%)"), ("test_acc", "Test accuracy (%)"), ("train_acc", "Train accuracy (%)"),
          ("training_loss", "Training loss (sum over batches)"), ("low_prob_ratio", "Low-probability ratio"),
          ("logit_std", "Logit std")]

def load_selected(name, model):
    path = sweep_path(name, model)
    if not os.path.exists(path):
        return None, None, None
    sweep = pd.read_csv(path, dtype={"lr": str, "alpha": str})
    chosen = sweep[sweep["selected"]].set_index("loss")
    order = [loss for loss in LOSSES if loss in chosen.index]
    per_run = pd.DataFrame({loss: pd.read_csv(os.path.join(chosen.loc[loss, "result_dir"], "summary.csv"))
                            .head(RUNS).set_index("run")["test_acc"] for loss in order})
    per_run.index.name = "run"
    return sweep, chosen.loc[order], per_run

def label(chosen, loss):
    r = chosen.loc[loss]
    return "{} (lr={}{})".format(loss, r["lr"], "" if pd.isna(r["alpha"]) else ", alpha=" + r["alpha"])

def show_results(name, model):
    sweep, chosen, per_run = load_selected(name, model)
    if sweep is None:
        print("no results for", name, model)
        return
    prefix = "{}_{}_{}_{}".format(name, DATASET, model, OPT_ALG)
    print("Sweep,", name, model)
    display(sweep.drop(columns="result_dir").style.format(precision=2, na_rep="")
            .apply(lambda r: ["font-weight: bold; background-color: #fff3bf" if r["selected"] else ""] * len(r), axis=1)
            .hide(axis="index"))

    base = per_run.columns[0]
    final = []
    for loss in per_run.columns:
        diff = (per_run[loss] - per_run[base]).dropna()
        p = stats.ttest_rel(per_run[loss], per_run[base], nan_policy="omit").pvalue if loss != base and len(diff) > 1 else np.nan
        final.append({"loss": label(chosen, loss), "runs": len(diff),
                      "test_acc": "{:.2f} ± {:.2f}".format(per_run[loss].mean(), per_run[loss].std()),
                      "Δ vs " + base: diff.mean(), "wins": "{}/{}".format((diff > 0).sum(), len(diff)), "p": p})
    final = pd.DataFrame(final)
    print("Comparison at the selected configurations,", name, model)
    display(final.style.format(precision=2, na_rep="").format({"p": "{:.3f}"}, na_rep="").hide(axis="index"))
    print("Test accuracy per run:")
    display(per_run.assign(**{loss + " - " + base: per_run[loss] - per_run[base] for loss in per_run.columns if loss != base})
            .style.format(precision=2))
    for loss, row in zip(per_run.columns, final.to_dict("records")):
        if loss == base:
            continue
        delta = row["Δ vs " + base]
        if np.isnan(row["p"]):
            verdict = "not enough runs for a test"
        elif row["p"] < 0.05:
            verdict = "significantly {} than {}".format("better" if delta > 0 else "worse", base)
        else:
            verdict = "no significant difference from {} (more runs may be needed)".format(base)
        print("{}: Δ = {:+.2f}, {}".format(loss, delta, verdict))
    final.to_csv(os.path.join("result", prefix + "_final.csv"), index=False)
    per_run.to_csv(os.path.join("result", prefix + "_per_run.csv"))

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    for loss in chosen.index:
        result_dir = chosen.loc[loss, "result_dir"]
        runs = [pd.read_csv(os.path.join(result_dir, "{}.csv".format(t))) for t in range(RUNS)
                if os.path.exists(os.path.join(result_dir, "{}.csv".format(t)))]
        curves = pd.concat(runs).groupby("epoch")
        mean, std = curves.mean(), curves.std().fillna(0)
        for ax, (col, _) in zip(axes.flat, CURVES):
            if col in mean:
                ax.plot(mean.index, mean[col], label=label(chosen, loss))
                ax.fill_between(mean.index, mean[col] - std[col], mean[col] + std[col], alpha=0.2)
    for ax, (_, title) in zip(axes.flat, CURVES):
        ax.set_title(title)
        ax.set_xlabel("epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(alpha=0.3)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle("{} / {} / {} / {}".format(name, DATASET, model, OPT_ALG))
    fig.tight_layout()
    fig.savefig(os.path.join("result", prefix + "_curves.png"), dpi=150)
    plt.show()

### 5.1 Stage 1: all losses, 30 epochs

In [ ]:
show_results(NAME, MODEL)

### 5.2 Stage 2: CE vs LWSCE by training budget

Each budget is a separate training with its own cosine schedule.

In [ ]:
rows = []
for epoch in SHORT_EPOCHS + [EPOCH]:
    name = NAME if epoch == EPOCH else "{}_e{}".format(NAME, epoch - 1)
    _, chosen, per_run = load_selected(name, MODEL)
    if per_run is None or not {"CROSSENTROPY", "LWSCE"} <= set(per_run.columns):
        print("missing budget", epoch - 1)
        continue
    diff = per_run["LWSCE"] - per_run["CROSSENTROPY"]
    rows.append({"epochs": epoch - 1, "LWSCE config": label(chosen, "LWSCE"),
                 "CE": "{:.2f} ± {:.2f}".format(per_run["CROSSENTROPY"].mean(), per_run["CROSSENTROPY"].std()),
                 "LWSCE": "{:.2f} ± {:.2f}".format(per_run["LWSCE"].mean(), per_run["LWSCE"].std()),
                 "Δ": diff.mean(), "wins": "{}/{}".format((diff > 0).sum(), len(diff)),
                 "p": stats.ttest_rel(per_run["LWSCE"], per_run["CROSSENTROPY"]).pvalue if len(diff) > 1 else np.nan})
budget = pd.DataFrame(rows)
if len(budget):
    display(budget.style.format(precision=2, na_rep="").format({"p": "{:.3f}"}, na_rep="").hide(axis="index"))
    budget.to_csv(os.path.join("result", "{}_{}_{}_{}_budget.csv".format(NAME, DATASET, MODEL, OPT_ALG)), index=False)
for epoch in SHORT_EPOCHS:
    show_results("{}_e{}".format(NAME, epoch - 1), MODEL)

### 5.3 Stage 3: ResNet-50

In [ ]:
show_results(NAME, "resnet50")

## 6. Download results (optional)

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/pilot_results", "zip", os.path.join(DRIVE_DIR, "result"))
files.download(archive)